In [1]:
import rasterio
import numpy as np
import os

In [2]:
past_ndwi_path = "../data/processed/past_ndwi.tif"
recent_ndwi_path = "../data/processed/recent_ndwi.tif"

with rasterio.open(past_ndwi_path) as src:
    past_ndwi = src.read(1)
    profile = src.profile
    transform = src.transform

with rasterio.open(recent_ndwi_path) as src:
    recent_ndwi = src.read(1)

print("NDWI files loaded")

NDWI files loaded


In [3]:
water_threshold = 0.3

past_water = past_ndwi > water_threshold
recent_water = recent_ndwi > water_threshold

print("Water masks generated")

Water masks generated


In [4]:
pixel_area_m2 = 30 * 30
pixel_area_km2 = pixel_area_m2 / 1_000_000

past_area = np.sum(past_water) * pixel_area_km2
recent_area = np.sum(recent_water) * pixel_area_km2

print("Past Water Area (sq km):", round(past_area, 2))
print("Recent Water Area (sq km):", round(recent_area, 2))

Past Water Area (sq km): 0.37
Recent Water Area (sq km): 2.83


In [5]:
increase = recent_area - past_area

if past_area != 0:
    percent_increase = (increase / past_area) * 100
else:
    percent_increase = 0

print("Water Increase (%):", round(percent_increase, 2))

Water Increase (%): 659.9


In [6]:
flood_expansion = np.logical_and(recent_water, np.logical_not(past_water))

profile.update(dtype=rasterio.uint8, count=1)

with rasterio.open("../data/processed/flood_expansion.tif", "w", **profile) as dst:
    dst.write(flood_expansion.astype(rasterio.uint8), 1)

print("Flood expansion map saved")

Flood expansion map saved
